# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 34.7 MB/s eta 0:00:00


In [ ]:
!pip install transformers

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 13.2 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# login

In [ ]:
import huggingface_hub
huggingface_hub.login(HiggingFace_TOKEN)

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import random
import pandas as pd
import re
import string
import sys
import argparse
import os
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
from sklearn.utils import shuffle
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split

import emoji
import pyarabic.araby as araby

In [ ]:
import torch
import torch.distributed as dist

# Define Evaluation Function

In [ ]:
def evaluate(y_true, y_pred):
    labels = ['إيجابية', 'سلبية', 'محايدة']
    mapping = {'محايدة': 0,
               'Neutral':0,
               'neutral':0,
               'سلبية': 1,
               'negative':1,
               'Negative':1,
               'إيجابية':2,
               'positive':2,
               'Positive':2,
               'none':3}
    def map_func(x):
        return mapping.get(x, 1)

    y_true = np.vectorize(map_func)(y_true)
    y_pred = np.vectorize(map_func)(y_pred)

    # Calculate accuracy
    accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
    print(f'Accuracy: {accuracy:.3f}')

    # Generate accuracy report
    unique_labels = set(y_true)  # Get unique labels

    f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
    print('\nf1_score: ', f1t)

    # Generate classification report
    class_report = classification_report(y_true=y_true,
                                         y_pred=y_pred,
                                         digits=4,
                                         target_names=['Neutral', 'Negative', 'Positive'])
    print('\nClassification Report:')
    print(class_report)

    # Generate confusion matrix
    conf_matrix = confusion_matrix(y_true=y_true, y_pred=y_pred, labels=[0, 1, 2])
    print('\nConfusion Matrix:')
    print(conf_matrix)

# Define Predict Function

In [ ]:
def predict(X_test, pipe):
    y_pred = []
    # Create a Hugging Face Dataset
    dataset = Dataset.from_pandas(X_test)

    # Use the pipeline with a dataset
    outputs = pipe(dataset["text"], batch_size=32,
                   pad_token_id=pipe.tokenizer.eos_token_id)

    # Process results
    results = [output[0]['generated_text'].lower() for output in outputs]

    # Extract the text after "answer:" for each result
    for i in tqdm(range(len(results))):
        answer_index = results[i].find("answer:") + len("answer:")
        extracted_text = results[i][answer_index:].strip()
        if ('إيجابي' or 'positive' or 'Positive' or 'pos') in extracted_text:
            y_pred.append("إيجابية")
        elif ('سلبي' or 'negative' or 'Negative' or 'neg') in extracted_text:
            y_pred.append("سلبية")
        elif ('محايد' or 'Neutral' or 'neutral' or 'neu') in extracted_text:
            y_pred.append("محايدة")
        else:
            y_pred.append("none")
    return y_pred

# Load the Data

In [ ]:
data = pd.read_excel('All Climate Change Data - All Related.xlsx')

data.drop_duplicates(subset='text', inplace = True)
data.dropna(inplace = True, subset='text')
data.reset_index(drop=True, inplace = True)

data.shape

(23957, 4)

# Preprocessing

In [ ]:
data['text'] = data['text'].str.replace(r'[^\w\s]+', ' ')
data['text'] = data['text'].str.replace("\s+", " ", regex=True)

data.head()

,location,date,text,sentiment
0,NaN,"2008, 10, 27, 17, 41, 17, tzinfo=datetime.time...",هذه ال٢.٥٪ لا تنطلق إلى الفضاء الكوني فتحتبس و...,Negative
1,Türkiye,"2023, 2, 20, 19, 2, 5, tzinfo=datetime.timezon...",#عاجل | إدارة الكوارث والطوارئ التركية: إزالة ...,Neutral
2,NaN,"2023, 2, 17, 9, 45, tzinfo=datetime.timezone.utc",RT @USUN: عُقد في مالطا هذا الأسبوع أول اجتماع...,Neutral
3,NaN,"2024, 8, 17, 16, 1, 17, tzinfo=datetime.timezo...",رغم ارتفاع درجات الحرارة وأشعة الشمس اللاهبة و...,Positive
4,NaN,"2024, 8, 11, 10, 32, 31, tzinfo=datetime.timez...",قبل أيام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Neutral


In [ ]:
data = data[['text', 'sentiment']]

In [ ]:
data['sentiment'].value_counts()

,count
sentiment,
Neutral,10307
Negative,8140
Positive,5510


In [ ]:
arabic_punctuations = '''`÷×؛<>_()*&^%][ـ،/:"؟.,'{}~¦+|!”…“–ـ'''
english_punctuations = string.punctuation
punctuations_list = arabic_punctuations + english_punctuations

def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("گ", "ك", text)
    return text

In [ ]:
data['text'] = data['text'].apply(normalize_arabic)

data.head()

,text,sentiment
0,هذه ال٢.٥٪ لا تنطلق الى الفضاء الكوني فتحتبس و...,Negative
1,#عاجل | ادارة الكوارث والطوارئ التركية: ازالة ...,Neutral
2,RT @USUN: عُقد في مالطا هذا الاسبوع اول اجتماع...,Neutral
3,رغم ارتفاع درجات الحرارة واشعة الشمس اللاهبة و...,Positive
4,قبل ايام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Neutral


In [ ]:
def data_cleaning(text):
    """Clean and preprocess text data.
    Args:
        text (pd.Series): A pandas Series containing text data to be cleaned.
    Returns:
        pd.Series: A pandas Series with the cleaned text data.

    Cleaning Steps:
    - Removes emojis and special characters like '\x89Û_', '&amp', etc.
    - Replaces consecutive dots with an empty string.
    - Removes '#' symbol from text.
    - Removes user names starting with '@'.
    - Removes URLs starting with 'http' or 'https'.
    - Remove diacritics.
    - Remove English.
    - fix elongated words
    - Normalize Characters
    - Removes extra whitespaces between words.

    """
    clean = text
    # Replace consecutive dots with an empty string
    pattern = re.compile('\\.+?(?=\B|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Replace '\x89Û_' with a whitespace
    pattern = re.compile('\x89Û_')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Replace newline characters with a whitespace
    pattern = re.compile('\\n')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Remove '#' symbol from text
    clean = clean.apply(lambda r: r.replace('#', ''))
    # Remove '_' symbol from text
    pattern = re.compile('_')
    clean = clean.apply(lambda r: re.sub(pattern, ' ', r))
    # Replace user names with '@'
    pattern = re.compile('@[a-zA-Z0-9\_]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='@'))
    # Remove URLs
    pattern = re.compile('https?\S+(?=\s|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='www'))
    # Remove emojis
    clean = clean.apply(lambda r: emoji.replace_emoji(r, replace=""))
    # Remove diacritics
    clean = clean.apply(lambda r: araby.strip_diacritics(r))
    # Remove English
    # pattern = re.compile(r'[a-zA-Z]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Fix elongated words
    pattern = re.compile(r'(.)\1{2,}')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=r'\1'))
    # Normalize letters
    pattern = re.compile("[إأآا]")
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl="ا"))
    # Remove extra whitespaces
    clean = clean.apply(lambda r: ' '.join(r.split()))  # Remove extra whitespaces between words

    return clean

In [ ]:
data['text'] = data_cleaning(data['text'])

In [ ]:
def remove_ids(text):
  return text.split("—")[0].strip()

data['text'] = data['text'].apply(remove_ids)
data.head()

,text,sentiment
0,هذه ال٢.٥٪ لا تنطلق الى الفضاء الكوني فتحتبس و...,Negative
1,عاجل | ادارة الكوارث والطوارئ التركية: ازالة ا...,Neutral
2,RT @: عقد في مالطا هذا الاسبوع اول اجتماع لمجل...,Neutral
3,رغم ارتفاع درجات الحرارة واشعة الشمس اللاهبة و...,Positive
4,قبل ايام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Neutral


In [ ]:
# Mapping dictionary
sentiment_map = {
    'Positive': 'إيجابية',
    'Negative': 'سلبية',
    'Neutral': 'محايدة'
}

# Apply the mapping
data['Final Label'] = data['sentiment'].map(sentiment_map)
data.head()

,text,sentiment,Final Label
0,هذه ال٢.٥٪ لا تنطلق الى الفضاء الكوني فتحتبس و...,Negative,سلبية
1,عاجل | ادارة الكوارث والطوارئ التركية: ازالة ا...,Neutral,محايدة
2,RT @: عقد في مالطا هذا الاسبوع اول اجتماع لمجل...,Neutral,محايدة
3,رغم ارتفاع درجات الحرارة واشعة الشمس اللاهبة و...,Positive,إيجابية
4,قبل ايام تم استضافتي لتسجيل بودكاست بحكم اختصا...,Neutral,محايدة


In [ ]:
data.drop_duplicates(subset='text', inplace = True)
data.dropna(inplace = True)
data.shape

# Split Data

In [ ]:
train, temp = train_test_split(data[['text', 'Final Label']],
                               test_size=0.3,
                               stratify=data['Final Label'],
                               random_state=42)

val, test = train_test_split(temp,
                             test_size=0.5,
                             stratify=temp['Final Label'],
                             random_state=42)

In [ ]:
train_indecies = train.index.values
val_indecies = val.index.values
test_indecies = test.index.values


data = {
    'train': train_indecies,
    'val': val_indecies,
    'test': test_indecies
}

max_len = max(len(train_indecies), len(val_indecies), len(test_indecies))
indecies = pd.DataFrame({k: pd.Series(v, dtype='float') for k, v in data.items()})

indecies = pd.DataFrame({
    'train': pd.Series(train_indecies),
    'val': pd.Series(val_indecies),
    'test': pd.Series(test_indecies)
})

indecies.to_csv('Train-Val-Test-Indecies-Climate.csv', index = False)

In [ ]:
train = train.reset_index(drop=True)

def generate_prompt(data_point):
    return f"""
    [INST] أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر المتعلقة بتغير المناخ.
    قم بتصنيف مشاعر الجملة العربية التالية  بين قوسين مربعين على اليسار بناءً على نبرتها العاطفية تجاه تغير المناخ. اختر مشاعر واحدة فقط من بين: إيجابية، سلبية، أو محايدة لهذه الجملة العربية.  [/INST]
    [{data_point["text"]}] = [{data_point["Final Label"]}]
            """.strip()

def generate_test_prompt(data_point):
    return f"""
    أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر المتعلقة بتغير المناخ.
    قم بتصنيف مشاعر الجملة العربية التالية بين قوسين مربعين على اليسار بناءً على نبرتها العاطفية تجاه تغير المناخ. اختر مشاعر واحدة فقط من بين: إيجابية، سلبية، أو محايدة لهذه الجملة العربية.
    [{data_point["text"]}] =
            """.strip()

train = pd.DataFrame(train.apply(generate_prompt, axis=1),
                       columns=["text"])
val = pd.DataFrame(val.apply(generate_prompt, axis=1),
                      columns=["text"])

In [ ]:
train.head()

,text
0,[INST] أنت مساعد ذكاء اصطناعي متخصص في تحليل ا...
1,[INST] أنت مساعد ذكاء اصطناعي متخصص في تحليل ا...
2,[INST] أنت مساعد ذكاء اصطناعي متخصص في تحليل ا...
3,[INST] أنت مساعد ذكاء اصطناعي متخصص في تحليل ا...
4,[INST] أنت مساعد ذكاء اصطناعي متخصص في تحليل ا...


In [ ]:
y_true = test["Final Label"]
X_test = pd.DataFrame(test.apply(generate_test_prompt, axis=1), columns=["text"])

train_data = Dataset.from_pandas(train)
eval_data = Dataset.from_pandas(train)

In [ ]:
X_test.shape

(3594, 1)

# Load Model

In [ ]:
import os

os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"
os.environ["PYTORCH_USE_CUDA_DSA"] = "1"

model_name = "core42/jais-13b"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map = {"": 0},
    quantization_config=bnb_config,
    trust_remote_code=True
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          use_fast=False,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=20,
                temperature=0.2
               )

Device set to use cuda:0


# Predict Without Fine-tuning

In [ ]:
max_length = tokenizer.model_max_length
for i in tqdm(range(len(X_test[:5]))):
    prompt = X_test.iloc[i]["text"]
    result = pipe(prompt, truncation = True, pad_token_id=tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("=")[-1].lower()
    print('\n')
    print(result)
    print('\n')
    print(answer)

In [ ]:
y_pred_b = predict(X_test, pipe)
evaluate(y_true, y_pred_b)

# Fine-tune Model

In [ ]:
from trl import SFTConfig

lora_target_modules = ["c_attn", "c_proj", "c_fc"]

peft_config = LoraConfig(
    lora_alpha=256,
    lora_dropout=0.05,
    r=128,
    target_modules=lora_target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)

training_arguments = SFTConfig(
    output_dir="logs",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=25,
    learning_rate=5e-5,
    weight_decay=0,
    fp16=False,
    bf16=False,
    max_grad_norm=1,
    max_steps=-1,
    warmup_ratio=0.1,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    eval_strategy="epoch",
    dataset_text_field="text",
    max_length=128
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_arguments
)

You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


Adding EOS to train dataset:   0%|          | 0/16771 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16771 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/16771 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/16771 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/16771 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/16771 [00:00<?, ? examples/s]

In [ ]:
# Train model
trainer.train()

# Save trained model
trainer.model.save_pretrained("/content/drive/MyDrive/Jais")

Epoch,Training Loss,Validation Loss
1,1.191200,0.995715


Epoch,Training Loss,Validation Loss
1,1.191200,0.995715
2,0.960200,0.740047
3,0.641200,0.611013


# Predict

In [ ]:
from peft import PeftModel, PeftConfig
# Load the Lora model
model = PeftModel.from_pretrained(model, "/content/drive/MyDrive/Jais")

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=20,
                temperature=0.2
               )

Device set to use cuda:0


In [ ]:
y_pred = []

for i in tqdm(range(len(X_test))):
    prompt = X_test.iloc[i]["text"]
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128).to("cuda")
    result = model.generate(
        **inputs,
        top_p=0.9,
        max_new_tokens=5,
        temperature=0.2,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.2,
        do_sample=True,
    )
    decoded = tokenizer.decode(result[0], skip_special_tokens=True)
    answer = decoded.split("=")[-1].lower()
    # print(answer)

    if ('إيجابي' or 'positive' or 'Positive' or 'pos' or 'إيجابيّة' or 'إيجابية') in answer:
        y_pred.append("إيجابية")
    elif ('سلبي' or 'negative' or 'Negative' or 'neg') in answer:
        y_pred.append("سلبية")
    elif ('محايد' or 'Neutral' or 'neutral' or 'neu') in answer:
        y_pred.append("محايدة")
    else:
        y_pred.append("none")

 84%|████████▎ | 3008/3594 [52:16<10:08,  1.04s/it]

In [ ]:
evaluate(y_true, y_pred)

Accuracy: 0.738

f1_score:  0.7368770764629112

Classification Report:
              precision    recall  f1-score   support

     Neutral     0.7303    0.6604    0.6936      1546
    Negative     0.7467    0.7895    0.7675      1221
    Positive     0.7392    0.8089    0.7725       827

    accuracy                         0.7385      3594
   macro avg     0.7388    0.7530    0.7445      3594
weighted avg     0.7379    0.7385    0.7369      3594


Confusion Matrix:
[[1021  303  222]
 [ 243  964   14]
 [ 134   24  669]]


# Pred on Asad

In [ ]:
asad2 = pd.read_csv('train_all.csv')

asad2['Text'] = asad2['Text'].str.replace(r'[^\w\s]+', '')
asad2['Text'] = asad2['Text'].str.replace("\s+", " ", regex=True)

asad2['Text'] = asad2['Text'].apply(normalize_arabic)

asad2['Text'] = data_cleaning(asad2['Text'])
asad2['Text'] = asad2['Text'].apply(remove_specific_term)
asad2['Text'] = asad2['Text'].apply(remove_ids)

asad2.dropna(inplace = True)
asad2 = asad2.drop_duplicates(subset = 'Text')
asad2 = asad2.rename(columns={'Text': 'text'})
asad2.head()

,Tweet_id,sentiment,text
0,1221884257490042887,neutral,الزعل بيغير ملامحك بيغير نظرة العين بيغير شكلك...
1,1221884400377499651,neutral,@ @ ليس حبا في ايران بقدر ماهو نكايه بترامب وحزبه
2,1221881406168731649,neutral,@ ابي اعرف الحاكم العربي المسلم اشلون ينام ماي...
3,1221882047691657217,neutral,@ @ في الخطاب تبع سليم سعاده حطت عالتويتر شو ق...
4,1221880371773673474,neutral,@ مفيش الكلام ده في الزمن


In [ ]:
train, asad20 = train_test_split(asad2,
                               test_size=20000,
                               stratify=asad2['sentiment'],
                               random_state=42)

asad20_test = pd.DataFrame(asad20.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:
asad2 = pd.read_csv('train_all.csv')

asad2['Text'] = asad2['Text'].str.replace(r'[^\w\s]+', '')
asad2['Text'] = asad2['Text'].str.replace("\s+", " ", regex=True)

asad2['Text'] = asad2['Text'].apply(normalize_arabic)

asad2['Text'] = data_cleaning(asad2['Text'])
asad2['Text'] = asad2['Text'].apply(remove_ids)

asad2.dropna(inplace = True)
asad2 = asad2.drop_duplicates(subset = 'Text')
asad2 = asad2.rename(columns={'Text': 'text'})
asad2.head()

,Tweet_id,sentiment,text
0,1221884257490042887,neutral,الزعل بيغير ملامحك بيغير نظرة العين بيغير شكلك...
1,1221884400377499651,neutral,@ @ ليس حبا في ايران بقدر ماهو نكايه بترامب وحزبه
2,1221881406168731649,neutral,@ ابي اعرف الحاكم العربي المسلم اشلون ينام ماي...
3,1221882047691657217,neutral,@ @ في الخطاب تبع سليم سعاده حطت عالتويتر شو ق...
4,1221880371773673474,neutral,@ مفيش الكلام ده في الزمن


In [ ]:
asad2['sentiment'] = asad2['sentiment'].map({
    'negative': 'سلبية',
    'neutral': 'محايدة',
    'positive': 'إيجابية'
})

In [ ]:
train, asad20 = train_test_split(asad2,
                               test_size=20000,
                               stratify=asad2['sentiment'],
                               random_state=42)

asad20_test = pd.DataFrame(asad20.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:
pred_asad20 = []

for i in tqdm(range(len(asad20_test))):
    prompt = asad20_test.iloc[i]["text"]
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128).to("cuda")
    result = model.generate(
        **inputs,
        top_p=0.9,
        max_new_tokens=5,
        temperature=0.2,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.2,
        do_sample=True,
    )
    decoded = tokenizer.decode(result[0], skip_special_tokens=True)
    answer = decoded.split("=")[-1].lower()
    # print(answer)

    if ('إيجابي' or 'positive' or 'Positive' or 'pos' or 'إيجابيّة' or 'إيجابية') in answer:
        pred_asad20.append("إيجابية")
    elif ('سلبي' or 'negative' or 'Negative' or 'neg') in answer:
        pred_asad20.append("سلبية")
    elif ('محايد' or 'Neutral' or 'neutral' or 'neu') in answer:
        pred_asad20.append("محايدة")
    else:
        pred_asad20.append("none")

100%|██████████| 20000/20000 [5:43:58<00:00,  1.03s/it]


In [ ]:
pred_asad20 = predict(asad20_test, pipe)
evaluate(asad20['sentiment'], pred_asad20)

Accuracy: 0.683

f1_score:  0.6858165978706182

Classification Report:
              precision    recall  f1-score   support

     Neutral     0.7884    0.7556    0.7716     13622
    Negative     0.4606    0.6327    0.5331      3248
    Positive     0.5322    0.4220    0.4708      3130

    accuracy                         0.6835     20000
   macro avg     0.5937    0.6035    0.5918     20000
weighted avg     0.6950    0.6835    0.6858     20000


Confusion Matrix:
[[10293  2204  1125]
 [ 1157  2055    36]
 [ 1606   203  1321]]


# ASTD

In [ ]:
import pandas as pd

df = pd.read_csv('Tweets.txt', delimiter='\t', header=None, names=['text', 'Label'])
df.head()

In [ ]:
df = df[df['Label']!='OBJ']
df['Label'] = df['Label'].map({
    'NEG': 'سلبية',
    'NEUTRAL': 'محايدة',
    'POS': 'إيجابية'
})

df.head()

In [ ]:
df['text'] = df['text'].str.replace(r'[^\w\s]+', '')
df['text'] = df['text'].str.replace("\s+", " ", regex=True)

df['text'] = df['text'].apply(normalize_arabic)

df['text'] = data_cleaning(df['text'])
df['text'] = df['text'].apply(remove_ids)

df.dropna(inplace = True)
df = df.drop_duplicates(subset = 'text')

astd_test = pd.DataFrame(df.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=7,
                temperature=0.2
               )

In [ ]:
pred = []

for i in tqdm(range(len(astd_test))):
    prompt = astd_test.iloc[i]["text"]
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128).to("cuda")
    result = model.generate(
        **inputs,
        top_p=0.9,
        max_new_tokens=5,
        temperature=0.2,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.2,
        do_sample=True,
    )
    decoded = tokenizer.decode(result[0], skip_special_tokens=True)
    answer = decoded.split("=")[-1].lower()
    # print(answer)

    if ('إيجابي' or 'positive' or 'Positive' or 'pos' or 'إيجابيّة' or 'إيجابية') in answer:
        pred.append("إيجابية")
    elif ('سلبي' or 'negative' or 'Negative' or 'neg') in answer:
        pred.append("سلبية")
    elif ('محايد' or 'Neutral' or 'neutral' or 'neu') in answer:
        pred.append("محايدة")
    else:
        pred.append("none")

100%|██████████| 3223/3223 [55:11<00:00,  1.03s/it]


In [ ]:
evaluate(df['Label'], pred)

Accuracy: 0.615

f1_score:  0.6310607979185554

Classification Report:
              precision    recall  f1-score   support

     Neutral     0.3729    0.6870    0.4834       805
    Negative     0.8141    0.6697    0.7349      1641
    Positive     0.8479    0.4234    0.5648       777
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.6146      3223
   macro avg     0.5087    0.4450    0.4458      3223
weighted avg     0.7120    0.6146    0.6311      3223


Confusion Matrix:
[[ 553  220   30    2]
 [ 513 1099   29    0]
 [ 417   31  329    0]
 [   0    0    0    0]]


# SemEval

In [ ]:
semeval = pd.read_csv('SemEval2017-task4-train.subtask-A.arabic.txt', delimiter='\t', header=None, names=['id', 'sentiment', 'text'])
semeval = semeval[['text', 'sentiment']]

semeval['sentiment'] = semeval['sentiment'].map({
    'negative': 'سلبية',
    'neutral': 'محايدة',
    'positive': 'إيجابية'
})
semeval.head()

,text,sentiment
0,إجبار أبل على التعاون على فك شفرة اجهزتها http...,إيجابية
1,RT @20fourMedia: #غوغل تتحدى أبل وأمازون بأجهز...,إيجابية
2,جوجل تنافس أبل وسامسونج بهاتف جديد https://t.c...,إيجابية
3,رئيس شركة أبل: الواقع المعزز سيصبح أهم من الطع...,إيجابية
4,ساعة أبل في الأسواق مرة أخرى https://t.co/dY2x...,محايدة


In [ ]:
semeval['text'] = semeval['text'].str.replace(r'[^\w\s]+', '')
semeval['text'] = semeval['text'].str.replace("\s+", " ", regex=True)

semeval['text'] = semeval['text'].apply(normalize_arabic)

semeval['text'] = data_cleaning(semeval['text'])
semeval['text'] = semeval['text'].apply(remove_ids)

semeval.dropna(inplace = True)
semeval = semeval.drop_duplicates(subset = 'text')

semeval_test = pd.DataFrame(semeval.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:
y_pred = []

for i in tqdm(range(len(semeval_test))):
    prompt = semeval_test.iloc[i]["text"]
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128).to("cuda")
    result = model.generate(
        **inputs,
        top_p=0.9,
        max_new_tokens=5,
        temperature=0.2,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.2,
        do_sample=True,
    )
    decoded = tokenizer.decode(result[0], skip_special_tokens=True)
    answer = decoded.split("=")[-1].lower()
    # print(answer)

    if ('إيجابي' or 'positive' or 'Positive' or 'pos' or 'إيجابيّة' or 'إيجابية') in answer:
        y_pred.append("إيجابية")
    elif ('سلبي' or 'negative' or 'Negative' or 'neg') in answer:
        y_pred.append("سلبية")
    elif ('محايد' or 'Neutral' or 'neutral' or 'neu') in answer:
        y_pred.append("محايدة")
    else:
        y_pred.append("none")

100%|██████████| 3281/3281 [56:22<00:00,  1.03s/it]


In [ ]:
evaluate(semeval['sentiment'], y_pred)

Accuracy: 0.603

f1_score:  0.5556324005141172

Classification Report:
              precision    recall  f1-score   support

     Neutral     0.5334    0.8986    0.6694      1450
    Negative     0.8276    0.5455    0.6575      1100
    Positive     0.6916    0.1012    0.1766       731
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.6026      3281
   macro avg     0.5131    0.3863    0.3759      3281
weighted avg     0.6673    0.6026    0.5556      3281


Confusion Matrix:
[[1303  115   30    2]
 [ 493  600    3    4]
 [ 647   10   74    0]
 [   0    0    0    0]]
